In [2]:
import pandas as pd
from pathlib import Path

# Paths 
base_path = Path("/Users/fayebalomenou/Downloads/FINALTHESIS") #__file__
input_folder = base_path / "floors-parquet"
output_folder = base_path / "RDF_triples"

FLOORS = [10, 11, 12]

# Prefixes 
prefixes = """@prefix saref: <https://saref.etsi.org/core/> .
@prefix s4bldg: <https://saref.etsi.org/saref4bldg/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix om: <https://www.wurvoc.org/vocabularies/om-1.8/> .
@prefix nu: <https://vu.nl/nu-building#> .

"""

#Properties
sensor_properties = { 
    "temperature": ("Temperature", "om:degree_Celsius", "xsd:float", "Temperature"),
    "humidity": ("Humidity", "om:percent", "xsd:float", "Humidity"),
    "eCO2": ("eCO2", "om:parts_per_million", "xsd:float", "estimated CO2 level"),
    "loudness": ("Loudness", "om:decibel", "xsd:float", "Noise level"),
    "general_purpose": ("PresenceDetected", None,"xsd:boolean", "People presence"),
    "illuminance": ("Illuminance", "om:lux", "xsd:float", "Illuminance"),
    "particulate_matter_2_5": ("ParticulateMatter_2_5", None, "xsd:float", "PM-2.5 level"),
    "smoke_density": ("SmokeDensity", None, "xsd:float", "Smoke density"),
    "volatile_organic_compound_level": ("VOCLevel", None, "xsd:float", "VOC level"),
    "TVOC": ("TVOC", None, "xsd:float", "Total VOC"),
    "color_b": ("ColorB", None, "xsd:float", "Blue Light Intensity"),
    "color_c": ("ColorC", None, "xsd:float", "Clear Light Intensity"),
    "color_r": ("ColorR",  None, "xsd:float", "Red Light Intensity"),
    "color_g": ("ColorG", None, "xsd:float", "Green Light Intensity"),
    "pressure": ("Pressure", "om:kilopascal", "xsd:float", "Air pressure"),
    "rssi": ("RSSI", None, "xsd:float", "Bluetooth RSSI"),
    "sound": ("Sound", None, "xsd:float", "Sound amplitude"),
    "voltage": ("Voltage", "om:volt", "xsd:float", "Battery voltage level"),
    "acc_mag": ("AccMag", None, "xsd:float", "Accelerometer magnitude"),
    "acc_x": ("AccX", None, "xsd:float", "Accelerometer X"),
    "acc_y": ("AccY", None, "xsd:float", "Accelerometer Y"),
    "acc_z": ("AccZ", None, "xsd:float", "Accelerometer Z"),
}

def write_unit(unit_iri):
    if unit_iri:
        return f"    saref:isMeasuredIn {unit_iri} ;\n"
    else:
        return "" #if None omit it
        
def get_active_columns(group): #returns only columns with non-NaN values
    active = []
    for col in sensor_properties:
        if col in group.columns and group[col].notna().any():
            active.append(col)
    return active


def write_properties(sensor_name, floor, room_iri, lines, active_cols): #writes only active properties
    for col in active_cols:
        rdf_name, unit_iri, xsd_type, label = sensor_properties[col]
        prop_iri = f"nu:{rdf_name}_{sensor_name}_floor{floor}"

        if xsd_type == "xsd:boolean":
            prop_class = "saref:State"
        else:
            prop_class = "saref:Property"

        lines.append(f"{prop_iri} a {prop_class} ;\n"
                     f"    saref:hasName \"{label}\" ;\n"
                     f"    saref:isPropertyOf nu:Room_{room_iri}_Feature .\n")

        
def write_timestamp_readings(group, sensor_name, floor, links, values): #Writes timestamps, NaN-skipping of no measurement
    for _, row in group.iterrows():
        timestamp = pd.to_datetime(row["time"])
        ts_str    = timestamp.strftime("%Y%m%dT%H%M%S")
        ts_iso    = timestamp.isoformat()
 
        for col, (rdf_name, unit_iri, xsd_type, _label) in sensor_properties.items():
            if col not in group.columns: #first checks if column exist (active column)
                continue
            
            val = row[col]
 
            if pd.isna(val): #if NaN values in column skip them
                continue
 
            prop_iri = f"nu:{rdf_name}_{sensor_name}_floor{floor}"
            val_iri  = f"nu:Val_{rdf_name}_{sensor_name}_floor{floor}_{ts_str}"
 
            if xsd_type == "xsd:boolean":
                val_str = "true" if int(val) == 1 else "false"
            else:
                val_str = str(float(val))
 
            links.append(f"{prop_iri} saref:hasPropertyValue {val_iri} .")
 
            values.append(f"{val_iri} a saref:PropertyValue ;\n"
                          f"    saref:hasValue \"{val_str}\"^^{xsd_type} ;\n"
                          f"{write_unit(unit_iri)}"
                          f"    saref:hasTimestamp \"{ts_iso}\"^^xsd:dateTime .\n")

            
def generate_floor_rdf(df: pd.DataFrame, floor: int) -> str:
    lines  = [prefixes]
    links  = []
    values = []

    floor_iri = f"nu:Floor_{floor}"

    # Building and floor, written once
    lines.append(f"nu:NU_Building a s4bldg:Building .\n")
    lines.append(f"{floor_iri} a s4bldg:BuildingSpace ;\n"
                 f"    s4bldg:isSpaceOf nu:NU_Building .\n")

    #1st pass collects active columns (properties) per sensor and all properties per room (from all its sensors)
    sensor_active_cols = {}  
    room_all_props = {}     
 
    for sensor_name, group in df.groupby("sensor"): #iterates over sensors alphabetically
        room_iri = group["room_number"].dropna().iloc[0]
        active_cols = get_active_columns(group)
        sensor_active_cols[sensor_name] = (active_cols, room_iri)
 
        if room_iri not in room_all_props:
            room_all_props[room_iri] = set()
 
        for col in active_cols:
            rdf_name = sensor_properties[col][0]
            room_all_props[room_iri].add(
                f"nu:{rdf_name}_{sensor_name}_floor{floor}")
            
    #2nd pass writes the rdf based on what was collected in 1st pass
    seen_rooms = set()
    for sensor_name, group in df.groupby("sensor"):
        active_cols, room_iri = sensor_active_cols[sensor_name]

        #Declare room as BuildingSpace
        #Declare Feature Of Interest
        if room_iri not in seen_rooms:
            lines.append(f"nu:Room_{room_iri} a s4bldg:BuildingSpace ;\n"
                         f"    s4bldg:isSpaceOf {floor_iri} ;\n"
                         f"    saref:hasFeatureOfInterest nu:Room_{room_iri}_Feature .\n")
            lines.append(f"nu:Room_{room_iri}_Feature a saref:FeatureOfInterest .\n")
 
            #FOI hasProperty is written once with all properties across all sensors 
            all_props = ", ".join(sorted(room_all_props[room_iri]))
            lines.append(f"nu:Room_{room_iri}_Feature saref:hasProperty {all_props} .\n")
            seen_rooms.add(room_iri)
        
        # Sensor triple (observes only its own active properties)
        observes_iris = " ;\n    saref:observes ".join(
            f"nu:{sensor_properties[col][0]}_{sensor_name}_floor{floor}"
            for col in active_cols)
 
        lines.append(f"nu:{sensor_name}_floor{floor} a saref:Sensor ;\n"
                     f"    saref:isContainedIn nu:Room_{room_iri} ;\n"
                     f"    saref:observes {observes_iris} .\n")

        # Property declarations (only the active ones) for each sensor
        write_properties(sensor_name, floor, room_iri, lines, active_cols)

        # Timestamp readings
        write_timestamp_readings(group, sensor_name, floor, links, values)

    # order: static structure -> links -> values
    lines.extend(links)
    lines.extend(values)

    return "\n".join(lines)


for floor in FLOORS:
    parquet_file = input_folder / f"floor-{floor}.parquet"

    if not parquet_file.exists():
        print(f"[skip] File not found: {parquet_file}")
        continue

    print(f"Processing Floor {floor}...")
    df = pd.read_parquet(parquet_file, engine="fastparquet") #fastparquet engine was used because the parquet file couldn't be read

    rdf_text = generate_floor_rdf(df, floor)

    out_dir = output_folder / f"Floor_{floor}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"floor_{floor}.ttl"

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(rdf_text)

    print(f"RDF written -> {out_path}")

Processing Floor 10...
RDF written -> /Users/fayebalomenou/Downloads/FINALTHESIS/RDF_triples/Floor_10/floor_10.ttl
Processing Floor 11...
RDF written -> /Users/fayebalomenou/Downloads/FINALTHESIS/RDF_triples/Floor_11/floor_11.ttl
Processing Floor 12...
RDF written -> /Users/fayebalomenou/Downloads/FINALTHESIS/RDF_triples/Floor_12/floor_12.ttl
